# Titanic Survival Prediction

Predicts passenger survival using a **Random Forest Classifier** trained on the [Kaggle Titanic dataset](https://www.kaggle.com/competitions/titanic).

## Notebook Structure

| Cell | Description |
|------|-------------|
| 1 | Load `train.csv` and `test.csv` |
| 2 | Impute missing values, encode categorical features |
| 3 | Train Random Forest (100 trees), evaluate on 20% validation split |
| 4 | Preprocess test data, generate predictions, save `submission.csv` |
| 5 | Visualise Titanic deaths by age and sex |

**Validation accuracy: ~81%**

In [ ]:
import pandas as pd

# Load train and test datasets from CSV files
train_data = pd.read_csv("train.csv")
test_data = pd.read_csv("test.csv")

# Inspect available feature columns
print(train_data.columns)

In [ ]:
# Fill missing Age values with the column mean to avoid dropping rows
train_data['Age'].fillna(train_data['Age'].mean(), inplace=True)
# Fill missing Embarked with the most common port
train_data['Embarked'].fillna(train_data['Embarked'].mode()[0], inplace=True)

# Encode Sex as binary: male=0, female=1
train_data['Sex'] = train_data['Sex'].map({'male': 0, 'female': 1})
# Encode Embarked as ordinal: C=0, Q=1, S=2
train_data['Embarked'] = train_data['Embarked'].map({'C': 0, 'Q': 1, 'S': 2})

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# Features selected for training — excludes Name, Ticket, Cabin due to high cardinality or missing data
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
X = train_data[features]
y = train_data['Survived']

# Hold out 20% of training data for validation
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Train a Random Forest with 100 decision trees
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Evaluate accuracy on the held-out validation set
accuracy = model.score(X_val, y_val)
print(f"Validation Accuracy: {accuracy}")

In [ ]:
# Apply the same preprocessing to test data before predicting
test_data['Age'].fillna(test_data['Age'].mean(), inplace=True)
test_data['Fare'].fillna(test_data['Fare'].mean(), inplace=True)
test_data['Sex'] = test_data['Sex'].map({'male': 0, 'female': 1})
test_data['Embarked'] = test_data['Embarked'].map({'C': 0, 'Q': 1, 'S': 2})

X_test = test_data[features]

# Generate survival predictions for the test set
predictions = model.predict(X_test)

# Save predictions in Kaggle submission format (PassengerId + Survived)
output = pd.DataFrame({'PassengerId': test_data['PassengerId'], 'Survived': predictions})
output.to_csv('submission.csv', index=False)
print("Submission file created!")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

sns.set_theme(style="darkgrid", palette="deep")

# Map encoded columns back to readable labels for display
train_data['Survival Status'] = train_data['Survived'].map({0: 'Died', 1: 'Survived'})
train_data['Sex Label'] = train_data['Sex'].map({0: 'Male', 1: 'Female'})

# Filter to only passengers who died
died_data = train_data[train_data['Survival Status'] == 'Died']

fig, ax = plt.subplots(figsize=(14, 6))

# Plot age distribution of deaths split by sex, using 5-year bins side by side
sns.histplot(
    data=died_data,
    x='Age',
    hue='Sex Label',
    binwidth=5,
    multiple='dodge',
    palette={'Male': '#4C72B0', 'Female': '#DD8452'},
    edgecolor='white',
    linewidth=0.8,
    alpha=0.9,
    ax=ax
)

ax.set_title('Titanic Deaths by Age & Sex', fontsize=18, fontweight='bold', pad=16)
ax.set_xlabel('Age', fontsize=13, labelpad=10)
ax.set_ylabel('Number of Deaths', fontsize=13, labelpad=10)
# Tick every 5 years to align with bin edges
ax.xaxis.set_major_locator(ticker.MultipleLocator(5))
ax.tick_params(axis='both', labelsize=11)

legend = ax.get_legend()
legend.set_title('Sex', prop={'size': 12, 'weight': 'bold'})
for text in legend.get_texts():
    text.set_fontsize(11)

plt.tight_layout()
plt.show()